# 168 — Proyecto: respuesta a incidentes de IA

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución de referencia

**Ejercicio 1.** La fase de **post-incidente** (y la de análisis) se apoya más en evidencia
reproducible: el análisis de causa raíz y la conversión del caso en regresión requieren poder
reproducir exactamente qué ocurrió; sin evidencia trazable, el post-mortem es especulación.


In [ ]:
result = run_lab("capstone", seed=168)
assert result["kind"] == "capstone"
assert result["evidence"]
show(result)


**Ejercicio 2.** Orden y fase:

```text
d -> Detección y análisis   (confirmar y clasificar severidad)
b -> Contención             (desactivar por flag, reversible)
e -> Erradicación           (parchear la causa raíz: allowlist)
a -> Recuperación           (regresión con el caso antes de re-abrir)
c -> Post-incidente         (causa raíz documentada + golden set)
```

**Ejercicio 3.** Reversibles y que preservan evidencia: (1) tomar un snapshot de los logs afectados
antes de tocar nada; (2) rotar/desactivar el sink de logging a un almacenamiento aislado y restringir
el acceso, en lugar de borrar. Acción a evitar: **borrar o truncar los logs** "para limpiar la fuga":
destruye la evidencia necesaria para el análisis de causa raíz y para las obligaciones legales de
reporte; la mitigación correcta es restringir acceso y redactar, no eliminar.

**Ejercicio 4 (esquema).** Detección: monitoreo de tasa de abstención/alucinación cae por debajo del
umbral o reporte de usuario (clases 164-165). Severidad: SEV1 (daño potencial a la salud). Contención:
activar respuesta segura por defecto / desactivar el dominio médico por flag (reversible). Erradicación:
reforzar grounding + umbral de abstención + derivación obligatoria a profesional (clase 165). Recuperación:
regresión con los casos peligrosos reproducidos antes de re-abrir (clase 158). Prevención: casos al
golden set y riesgo reevaluado en la matriz (clases 158, 166); model card actualizada (clase 167).


In [ ]:
# Verificación del Ejercicio 2
acciones = {"a": "regresión", "b": "flag", "c": "post-mortem", "d": "detección", "e": "parche"}
orden = ["d", "b", "e", "a", "c"]
fases = ["Detección/análisis", "Contención", "Erradicación", "Recuperación", "Post-incidente"]
for k, f in zip(orden, fases):
    print(f"{k}: {f}")
assert orden == ["d", "b", "e", "a", "c"]


## Reflexión (guía)

1. Porque determinan si se puede responder (preparación) y si el incidente se repite (post-incidente),
   pero no dan resultados visibles inmediatos, así que se posponen bajo presión.
2. Su impacto real sobre personas o su relevancia legal/regulatoria; un bug sin daño ni obligación de
   reporte no es un incidente en este sentido.
3. Convierte el caso en un ítem permanente de regresión y en un riesgo catalogado con mitigación, de
   modo que la misma causa raíz se detecta y bloquea automáticamente en el futuro.
